# ゼロから作るショアのアルゴリズム 第2回: 剰余加算器

[第1回](ja/401_shor_scratch_adder.ipynb)では、通常の量子加算器を作りました。今回はその上に**剰余**加算 $a+b \bmod N$ を実装します — $x^j \bmod N$ を計算する過程で生じるすべての中間値は $[0, N)$ の範囲に収める必要があるため、これがショアのアルゴリズムの中で実際に必要とされる演算です。

参考文献: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995), 第III.B節 (Figure 4)

## 5ステップのアイデア

$(a+b) \bmod N$ は古典的には簡単です: 足し算して、結果が $N$ 以上ならその分だけ $N$ を引きます。しかし量子コンピュータでは、通常のように「確認してから分岐する」ことはできません(それには測定が必要になり、大事にしたい重ね合わせが壊れてしまいます)。代わりに、確認は*量子ビットに記録し*、条件分岐は*回路の形で*表現する必要があります。

1. **足し算**: $b \leftarrow a + b$(第1回の加算器を使用)
2. **$N$ を引く**: $b \leftarrow b - N$(加算器を逆向きに実行)
3. **符号を検出**: ステップ2で $b$ が負になった場合、$b$ レジスタの一番上の余分な量子ビットが $1$ になります(借り/オーバーフローのフラグとして働きます) — このビットをフラグ用量子ビット $t$ にコピーします
4. **条件付きで $N$ を足し戻す**: $t=1$ の場合(つまり $a+b < N$ であり、$N$ を引きすぎた場合)、$N$ を足し戻します — ただし重ね合わせの中で $t=1$ になっている枝*だけ*に対して
5. **後片付け**: $t$ を $\ket 0$ に戻し、符号検出のステップを元に戻します。これにより回路全体が、$b$ 以外には何も影響を残さないクリーンで再利用可能な操作になります

前提は $0 \le a, b < N$ です。ステップ1〜2は加算器/減算器をそのまま使います。**面白いのはステップ4**で、ここには論文が使っている少し意外な手口があります。

In [ ]:
!pip install git+https://github.com/blueqat/blueqatSDK

## 第1回の部品を再利用する

前回と同じ `carry`/`carry_inv`/`sum_gate` に加え、それらを組み立てた加算器全体 `add_circuit`、そしてその逆演算 `sub_circuit` を用意します。`sub_circuit` は、同じゲート列を単に逆順に実行するだけで得られます(ここで使われているゲートはすべて `CX` か `CCX` であり、どちらも自己逆元なので、順序を逆にするだけで十分です — 個々のゲートを反転させる必要はありません)。論文も同じことを書いています(*an adder whose network is run backwards*)。

In [2]:
from blueqat import Circuit

def carry(c, a, b, c_next):
    circ = Circuit()
    circ.ccx[a, b, c_next]
    circ.cx[a, b]
    circ.ccx[c, b, c_next]
    return circ

def carry_inv(c, a, b, c_next):
    circ = Circuit()
    circ.ccx[c, b, c_next]
    circ.cx[a, b]
    circ.ccx[a, b, c_next]
    return circ

def sum_gate(c, a, b):
    circ = Circuit()
    circ.cx[a, b]
    circ.cx[c, b]
    return circ

def add_circuit(c, a, b, n):
    """b := a + b, using carry register c. b must have n+1 qubits."""
    circ = Circuit()
    for i in range(n - 1):
        circ += carry(c[i], a[i], b[i], c[i + 1])
    circ += carry(c[n - 1], a[n - 1], b[n - 1], b[n])
    circ.cx[a[n - 1], b[n - 1]]
    circ += sum_gate(c[n - 1], a[n - 1], b[n - 1])
    for i in range(n - 2, -1, -1):
        circ += carry_inv(c[i], a[i], b[i], c[i + 1])
        circ += sum_gate(c[i], a[i], b[i])
    return circ

def sub_circuit(c, a, b, n):
    """b := b - a: the adder's gate list, reversed."""
    add_circ = add_circuit(c, a, b, n)
    reversed_circ = Circuit()
    reversed_circ.ops = list(reversed(add_circ.ops))
    return reversed_circ

## ステップ4: 加算器ではなく「足す数」のほうを条件付きにする

「$t=1$ のときだけ $b \leftarrow b + N$ を実行する」をどう作るか。

素直に考えると、`add_circuit(c, N, b, n)` に含まれる**すべてのゲートを $t$ で制御**することになります。`CX` をもう1つの量子ビットで制御すれば `CCX` になり簡単ですが、`CCX` をさらに制御すると**3制御X**が必要になり、このSDKには直接対応するゲートがありません。補助量子ビットとトフォリの連鎖で代用することはできますが、回路がかなり重くなります。

論文はまったく違うことをやっています。

> Conditionally on the value of this last qubit $\ket t$, $N$ is added back to the second register, leaving it with the value $a + b \bmod N$. This is done by **either leaving the first register with the value $N$** (in case of overflow), **or resetting it to 0** (if there is no overflow) **and then using a plain adder**.

つまり、**加算器には一切手を触れません**。代わりに、足す数を置いている $N$ レジスタのほうを条件付きにします。$t=1$ なら $N$ のまま、$t=0$ なら $0$ にリセットして、**無制御の加算器**をそのまま通す。

なぜこれで済むのか。**$0$ を足す加算は何もしないから**です。加算器そのものは無条件に走らせてよく、入力が $0$ なら勝手に恒等変換になってくれます。

この作り方だと、回路全体を通して**3制御ゲートが一度も出てきません**。補助量子ビットも不要です。

$N$ は回路を組む時点で分かっている古典的な定数なので、どのビットが $1$ かは既知です。$t$ を `X` で挟んだ $0$ 制御の CNOT を、$N$ の立っているビットに当てるだけで済みます。

In [3]:
def set_to_zero_if(t, N, val_N, n):
    """Reset the N register to 0 when t == 0 (leave it alone when t == 1).
    Self-inverse: applying it twice restores N."""
    circ = Circuit()
    circ.x[t]
    for j in range(n):
        if (val_N >> j) & 1:
            circ.cx[t, N[j]]
    circ.x[t]
    return circ

`X` も `CX` も自己逆元で、しかも互いにターゲットが異なるため、**この回路をもう一度流すだけで $N$ が元に戻ります**。「かけて、使って、同じものでほどく」という、第1回から繰り返し出てくるアンコンピュートの最も簡単な形です。

## レジスタ構成

剰余加算器全体には $4n+2$ 個の量子ビットが必要です。

- `c[0..n-1]`: 桁上げレジスタ(足し算・引き算のステップで共用)
- `a[0..n-1]`: 1つ目の加数
- `b[0..n]`: 2つ目の加数・結果を保持するレジスタ(第1回と同じ $n+1$ 幅で、ステップ3ではその最上位ビットがオーバーフロー/借りのフラグを兼ねる)
- `N[0..n-1]`: 法(定数として読み込む)
- `t`: 「$N$ を足し戻す必要があったか」を表すフラグ用量子ビット1個

多重制御ゲート用の補助量子ビットは要りません。

In [4]:
def modulo_add_circuit(c, a, b, N, t, n, val_N):
    """b := (a + b) mod N on the given registers -- the five steps, paper Fig.4.
    Carries no input encoding, so it can be reused as a black box."""
    circ = Circuit()

    # Steps 1-2: b <- a + b, then b <- b - N
    circ += add_circuit(c, a, b, n)
    circ += sub_circuit(c, N, b, n)

    # Step 3: copy the borrow/overflow bit into the flag t
    circ.cx[b[n], t]

    # Step 4 (paper, Fig.4): blank the N register when t == 0, then a PLAIN adder,
    # then restore N with the very same gates.
    gate = set_to_zero_if(t, N, val_N, n)
    circ += gate
    circ += add_circuit(c, N, b, n)
    circ += gate

    # Step 5: clean up -- undo the borrow/overflow detection and reset t to |0>
    circ += sub_circuit(c, a, b, n)
    circ.x[b[n]]
    circ.cx[b[n], t]
    circ.x[b[n]]
    circ += add_circuit(c, a, b, n)

    return circ


def build_modulo_adder(input_a, input_b, input_N, n):
    """Encode the inputs, then run the modular adder.
    Returns (circuit, b_register_qubit_indices)."""
    circ = Circuit(4 * n + 2)

    c = list(range(n))
    a = list(range(n, 2 * n))
    b = list(range(2 * n, 3 * n + 1))
    N = list(range(3 * n + 1, 4 * n + 1))
    t = 4 * n + 1

    for i in range(n):
        if (input_a >> i) & 1:
            circ.x[a[i]]
        if (input_b >> i) & 1:
            circ.x[b[i]]
        if (input_N >> i) & 1:
            circ.x[N[i]]

    circ += modulo_add_circuit(c, a, b, N, t, n, input_N)
    return circ, b

## テストする

第1回と同じビット順序の規則を使います: 結果文字列の中で量子ビット `q` は `state[total - 1 - q]` です(右端が0番目の量子ビット)。

シミュレータのバックエンドには `statevector` を明示します。blueqat のデフォルトはテンソルネットワークの縮約で動くため、コストが量子ビット数ではなく*回路のつながり方*で決まります。この種の「量子ビットは少ないが深い」回路とは相性が悪く、第3回の乗算器では20量子ビット足らずでメモリを使い切って落ちました。状態ベクトルなら消費メモリは $2^{(\text{量子ビット数})}$ で素直に決まります。

In [5]:
def run_modulo_adder(val_a, val_b, val_N, n=None):
    if n is None:
        n = max(val_N, 1).bit_length()
    circuit, b = build_modulo_adder(val_a, val_b, val_N, n)
    state = circuit.m[:].run(shots=1, backend="statevector").most_common(1)[0][0]
    total = circuit.n_qubits
    bits = [state[total - 1 - q] for q in b]  # bits[0] = LSB
    return int("".join(reversed(bits)), 2)

tests = [
    (7, 6, 11, 4),
    (14, 13, 15, 4),
    (3, 5, 9, 4),
    (0, 0, 7, 3),
    (5, 5, 6, 3),
]
for x, y, N, n in tests:
    result = run_modulo_adder(x, y, N, n)
    expected = (x + y) % N
    print(f"({x}+{y}) mod {N} = {result}  (expected {expected}, {'OK' if result == expected else 'MISMATCH'})")

(7+6) mod 11 = 2  (expected 2, OK)


(14+13) mod 15 = 12  (expected 12, OK)


(3+5) mod 9 = 8  (expected 8, OK)


(0+0) mod 7 = 0  (expected 0, OK)


(5+5) mod 6 = 4  (expected 4, OK)


いくつかは両方の分岐を通るように選んでいます: `(7+6) mod 11` は $N$ を引いても負にならない場合です($7+6=13 \ge 11$)。$t=0$ なので、ステップ4では $N$ レジスタが $0$ にされ、足し戻しは何も起こりません。一方 `(0+0) mod 7` は $N$ を引くと負になる場合で($0+0 < 7$)、$t=1$ となり、ステップ4で $N$ が足し戻されます — どちらの場合も、同じ回路で正しく動作します。

## 網羅テスト: 作業用の量子ビットが $\ket 0$ に戻っているか

剰余加算器は次回以降、何十回も繰り返し使う部品です。結果 $b$ が正しいだけでは足りません。**キャリーレジスタ $c$・法レジスタ $N$・フラグ $t$ がすべて元に戻っている**必要があります。戻っていないと、そのビットが結果と絡んだまま残り、後段の量子干渉が壊れてしまいます。

$N = 5, 7, 9$ について $a$ と $b$ を全通り試し、値と後片付けの両方を確認します。

In [6]:
def run_and_check(val_a, val_b, val_N, n=None):
    """Returns (result, everything_else_is_back_to_|0>)."""
    if n is None:
        n = max(val_N, 1).bit_length()
    circuit, b = build_modulo_adder(val_a, val_b, val_N, n)
    c = list(range(n))
    N = list(range(3 * n + 1, 4 * n + 1))
    t = 4 * n + 1
    state = circuit.m[:].run(shots=1, backend="statevector").most_common(1)[0][0]
    total = circuit.n_qubits
    read = lambda qs: int("".join(reversed([state[total - 1 - q] for q in qs])), 2)
    clean = read(N) == val_N and int(state[total - 1 - t]) == 0 and read(c) == 0
    return read(b), clean

bad = 0
for N in (5, 7, 9):
    for a in range(N):
        for b in range(N):
            result, clean = run_and_check(a, b, N)
            if result != (a + b) % N or not clean:
                bad += 1
                print(f"MISMATCH ({a}+{b}) mod {N} -> {result}, clean={clean}")
print(f"155 cases checked, {bad} mismatches")

155 cases checked, 0 mismatches


## 規模

量子ビット数もゲート数も $n$ に対して**線形**です。そして最大制御数はどこまでいっても $2$、つまりトフォリ止まりです。

In [7]:
for n in (3, 4, 8, 16):
    c = list(range(n)); a = list(range(n, 2 * n)); b = list(range(2 * n, 3 * n + 1))
    N = list(range(3 * n + 1, 4 * n + 1)); t = 4 * n + 1
    circ = modulo_add_circuit(c, a, b, N, t, n, (1 << n) - 1)
    max_ctrl = max(g.n_qargs - 1 for g in circ.ops)   # X=0, CX=1, CCX=2
    print(f"n={n:2d}: {4 * n + 2:3d} qubits, {len(circ.ops):4d} gates, "
          f"max {max_ctrl} controls")

n= 3:  14 qubits,  124 gates, max 2 controls
n= 4:  18 qubits,  166 gates, max 2 controls
n= 8:  34 qubits,  334 gates, max 2 controls
n=16:  66 qubits,  670 gates, max 2 controls


論文が "the number of elementary gates in the plain adder, the modular addition and the controlled–modular addition network scales linearly with $n$" と書いているとおりになりました。

## 次回予告

足し算と剰余加算ができたので、次のステップは[制御剰余乗算](ja/403_shor_scratch_controlled_multiplier.ipynb)です — ある定数を、量子ビットで制御しながら $N$ を法として掛ける操作を、被乗数をシフトしたコピーを繰り返し剰余加算することで実現します(可逆的に行う筆算式の掛け算です)。

そこでも、今回ステップ4で使ったのとまったく同じ手口 — **演算を制御するのではなく、入力を条件付きにして演算を自明にする** — がもう一度出てきます。論文はこの発想を Fig.4 と Fig.5 の2箇所で使っています。